In [1]:
import json
import numpy as np
import h5py
import xgboost as xgb
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

In [2]:
# ── Shared config ────────────────────────────────────────────────────────────
SPLITS_PATH = "/code/jjiang23/pathml/aim2_balanceV2/data/splits.json"
PHASE_NAMES = ["Phase1", "Phase2", "Phase3", "Phase4"]

LABEL_MAP = {
    b"Phase1": 0,
    b"Phase2": 1,
    b"Phase3": 2,
    b"Phase4": 3,
    b"nonphase": 4,   # excluded
}

XGB_PARAMS = dict(
    objective        = "multi:softprob",
    num_class        = 4,
    n_estimators     = 400,
    max_depth        = 6,
    learning_rate    = 0.1,
    subsample        = 0.8,
    colsample_bytree = 0.8,
    use_label_encoder= False,
    eval_metric      = "mlogloss",
    tree_method      = "hist",
    verbosity        = 0,
    n_jobs           = -1,
    random_state     = 42,
)

# ── Experiment configurations ────────────────────────────────────────────────
# joint_indices=None  → use all joints
# For MediaPipe (world_mp_cropped_iou): h5 shape (T, 1, J, >=3), J=33
#   joints 0-10  = face
#   joints 11-24 = upper body (shoulders → hips)
#   joints 23-32 = lower body (hips → feet)
# For MotionBert (motionBert_cropped_iou): h5 shape (T, J, D), J=17 (H3.6M)
#   joints 0-6   = lower body (pelvis, hips, knees, ankles)
#   joints 7-16  = upper body (spine, thorax, neck, head, shoulders, elbows, wrists)

EXPERIMENTS = [
    {
        "name":          "MPW Full (33J)",
        "h5_key":        "world_mp_cropped_iou",
        "joint_indices": None,
    },
    {
        "name":          "MPW Bottom Half (10J)",
        "h5_key":        "world_mp_cropped_iou",
        "joint_indices": list(range(23, 33)),   # hips → feet
    },
    {
        "name":          "MotionBert Full (17J)",
        "h5_key":        "motionBert_cropped_iou",
        "joint_indices": None,
    },
    {
        "name":          "MotionBert Bottom Half (7J)",
        "h5_key":        "motionBert_cropped_iou",
        "joint_indices": list(range(0, 7)),     # pelvis, hips, knees, ankles
    },
    {
        "name":          "MotionBert Top Half (10J)",
        "h5_key":        "motionBert_cropped_iou",
        "joint_indices": list(range(7, 17)),    # spine → wrists
    },
]

with open(SPLITS_PATH) as f:
    splits = json.load(f)

print(f"Folds: {list(splits.keys())}")
print(f"Experiments: {[e['name'] for e in EXPERIMENTS]}")

Folds: ['fold_0', 'fold_1', 'fold_2', 'fold_3', 'fold_4']
Experiments: ['MPW Full (33J)', 'MPW Bottom Half (10J)', 'MotionBert Full (17J)', 'MotionBert Bottom Half (7J)', 'MotionBert Top Half (10J)']


In [3]:
# ── Data loader ──────────────────────────────────────────────────────────────

def load_phase_frames(h5_paths, h5_key, joint_indices=None):
    """
    Load per-frame features (phase frames only, label 0-3) from h5 files.

    Args:
        h5_paths:      list of .h5 file paths
        h5_key:        dataset key inside each h5 file
        joint_indices: optional list of joint indices to keep (None = all)

    Returns:
        X: (N, J*3) float32
        y: (N,) int  — phase label 0-3
    """
    Xs, ys = [], []
    for path in h5_paths:
        try:
            with h5py.File(path, 'r') as f:
                if h5_key not in f:
                    continue
                raw = f[h5_key][:]
                # MPW: (T, 1, J, >=3)  — has person-index dim
                # MB:  (T, J, D)       — no person-index dim
                if raw.ndim == 4:
                    kp = raw[:, 0, :, :3].astype(np.float32)   # (T, J, 3)
                else:
                    kp = raw[:, :, :3].astype(np.float32)       # (T, J, 3)
                raw_labels = f['camera_poses_labels'][:]
        except Exception as e:
            print(f"  Skipping {path}: {e}")
            continue

        labels = np.array([LABEL_MAP[lbl] for lbl in raw_labels], dtype=np.int32)
        kp     = np.nan_to_num(kp)

        # Optional joint subset
        if joint_indices is not None:
            kp = kp[:, joint_indices, :]   # (T, len(joint_indices), 3)

        # Keep phase frames only
        mask = labels < 4
        if mask.sum() == 0:
            continue

        T, J, D = kp[mask].shape
        Xs.append(kp[mask].reshape(T, J * D))
        ys.append(labels[mask])

    if not Xs:
        return np.empty((0, 0), dtype=np.float32), np.empty((0,), dtype=np.int32)
    return np.concatenate(Xs, axis=0), np.concatenate(ys, axis=0)


# Sanity check
for exp in EXPERIMENTS:
    X, y = load_phase_frames(splits['fold_0']['train'][:2], exp['h5_key'], exp['joint_indices'])
    print(f"{exp['name']:<30} X={X.shape}  dist={np.bincount(y) if len(y) else 'empty'}")

MPW Full (33J)                 X=(1803, 99)  dist=[600 601 602]
MPW Bottom Half (10J)          X=(1803, 30)  dist=[600 601 602]
MotionBert Full (17J)          X=(1803, 51)  dist=[600 601 602]
MotionBert Bottom Half (7J)    X=(1803, 21)  dist=[600 601 602]
MotionBert Top Half (10J)      X=(1803, 30)  dist=[600 601 602]


In [ ]:
# ── Run all experiments × all folds ─────────────────────────────────────────
all_results = {}   # exp_name → list of fold result dicts

for exp in EXPERIMENTS:
    exp_name = exp['name']
    print(f"\n{'#'*70}")
    print(f"  EXPERIMENT: {exp_name}")
    print(f"{'#'*70}")

    fold_results = []

    for fold_name, fold_data in splits.items():
        print(f"\n  {'='*56}")
        print(f"  {fold_name}")
        print(f"  {'='*56}")

        X_train, y_train = load_phase_frames(fold_data['train'], exp['h5_key'], exp['joint_indices'])
        X_val,   y_val   = load_phase_frames(fold_data['val'],   exp['h5_key'], exp['joint_indices'])

        print(f"  train: {X_train.shape[0]:,} frames  val: {X_val.shape[0]:,} frames  "
              f"features: {X_train.shape[1] if len(X_train) else 0}")

        if X_train.shape[0] == 0 or X_val.shape[0] == 0:
            print("  Skipping — empty split")
            continue

        clf = xgb.XGBClassifier(**XGB_PARAMS)
        clf.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)

        y_pred = clf.predict(X_val)
        acc    = accuracy_score(y_val, y_pred)
        print(f"  Accuracy: {acc:.4f}")
        print(classification_report(y_val, y_pred, target_names=PHASE_NAMES, digits=3))

        fold_results.append({
            "fold": fold_name,
            "acc":  acc,
            "y_val":  y_val,
            "y_pred": y_pred,
            "cm":     confusion_matrix(y_val, y_pred),
        })

    all_results[exp_name] = fold_results


######################################################################
  EXPERIMENT: MPW Full (33J)
######################################################################

  fold_0
  train: 84,773 frames  val: 19,972 frames  features: 99
  Accuracy: 0.6630
              precision    recall  f1-score   support

      Phase1      0.637     0.689     0.662      5988
      Phase2      0.583     0.534     0.558      5767
      Phase3      0.665     0.697     0.681      5119
      Phase4      0.866     0.796     0.830      3098

    accuracy                          0.663     19972
   macro avg      0.688     0.679     0.683     19972
weighted avg      0.664     0.663     0.663     19972


  fold_1
  train: 83,325 frames  val: 21,420 frames  features: 99
  Accuracy: 0.7576
              precision    recall  f1-score   support

      Phase1      0.707     0.805     0.753      6333
      Phase2      0.697     0.563     0.623      5768
      Phase3      0.790     0.836     0.813      5711
    

In [ ]:
# ── Cross-experiment accuracy summary ────────────────────────────────────────
summary_rows = []
for exp_name, fold_results in all_results.items():
    accs = [r['acc'] for r in fold_results]
    if accs:
        summary_rows.append({
            'Experiment': exp_name,
            'Mean Acc':   np.mean(accs),
            'Std Acc':    np.std(accs),
            'Min':        np.min(accs),
            'Max':        np.max(accs),
            'N Folds':    len(accs),
        })

df_summary = pd.DataFrame(summary_rows).set_index('Experiment')
print(df_summary.to_string(float_format=lambda x: f'{x:.4f}'))

# Bar chart
fig, ax = plt.subplots(figsize=(9, 4))
means = df_summary['Mean Acc']
stds  = df_summary['Std Acc']
bars  = ax.bar(means.index, means.values, yerr=stds.values, capsize=5,
                color=plt.cm.tab10(np.linspace(0, 0.9, len(means))), alpha=0.85)
for bar, m in zip(bars, means.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{m:.3f}', ha='center', va='bottom', fontsize=9)
ax.set_ylabel('Accuracy')
ax.set_title('XGBoost Phase Classification — Skeleton Configuration Comparison (5-fold)')
ax.set_ylim(0, 1.05)
plt.xticks(rotation=15, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# ── Pooled confusion matrices per experiment ─────────────────────────────────
n_exp = len(all_results)
ncols = min(n_exp, 3)
nrows = (n_exp + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(5.5 * ncols, 4.5 * nrows))
axes = np.array(axes).flatten()

for ax, (exp_name, fold_results) in zip(axes, all_results.items()):
    if not fold_results:
        ax.set_visible(False)
        continue
    y_val_all  = np.concatenate([r['y_val']  for r in fold_results])
    y_pred_all = np.concatenate([r['y_pred'] for r in fold_results])
    cm = confusion_matrix(y_val_all, y_pred_all)
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    acc = accuracy_score(y_val_all, y_pred_all)

    sns.heatmap(
        cm_norm, annot=True, fmt='.2f', ax=ax,
        xticklabels=PHASE_NAMES, yticklabels=PHASE_NAMES,
        cmap='Blues', vmin=0, vmax=1, cbar=False,
    )
    ax.set_title(f'{exp_name}\nPooled Acc={acc:.3f}', fontsize=9)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')

for ax in axes[n_exp:]:
    ax.set_visible(False)

plt.suptitle('Pooled Confusion Matrices — Phase Classification (nonphase excluded)', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── Per-fold accuracy line plot ───────────────────────────────────────────────
fold_names = list(splits.keys())
fig, ax = plt.subplots(figsize=(9, 4))
colors = plt.cm.tab10(np.linspace(0, 0.9, len(all_results)))

for color, (exp_name, fold_results) in zip(colors, all_results.items()):
    if not fold_results:
        continue
    fold_accs = {r['fold']: r['acc'] for r in fold_results}
    ys = [fold_accs.get(fn, np.nan) for fn in fold_names]
    ax.plot(fold_names, ys, marker='o', label=exp_name, color=color)

ax.set_ylabel('Accuracy')
ax.set_title('Per-fold Accuracy by Skeleton Configuration')
ax.set_ylim(0, 1.05)
ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=8)
plt.tight_layout()
plt.show()